# Retrieval-Augmented Fine-Tuning (RAFT)

Companion notebook for the [RAFT lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/14-retrieval-augmented-fine-tuning).

**The idea in one sentence.** RAFT *fine-tunes* a model on the RAG task in your domain: each training example pairs a question with a **golden** document plus **distractor** documents, and a fraction of examples contain *only* distractors — so the model learns to use retrieved context, cite it, and stay robust when retrieval misses.

We build the RAFT data recipe from scratch, simulate why the golden-inclusion probability **P** matters, sketch the library training path, then cover tradeoffs and a hands-on exercise.

> **To save your work:** click **Copy to Drive** at the top, or File -> Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## 1. The RAFT data recipe, from scratch

A tiny domain corpus. For each question we know the **golden** chunk that answers it; every other chunk is a potential **distractor**. RAFT assembles training examples by mixing them.

In [ ]:
CORPUS = {
    'refund_window': 'Refunds are accepted within 30 days of purchase.',
    'refund_how':    'To request a refund, email support with your order number.',
    'shipping_std':  'Standard shipping takes 3 to 5 business days.',
    'shipping_intl': 'International shipping can take up to 2 weeks.',
    'password':      'Reset your password from the account settings page.',
}
QA = [
    ('How long do I have to get a refund?', 'refund_window', '30 days'),
    ('How do I request a refund?',          'refund_how',    'email support with your order number'),
    ('How long is standard shipping?',      'shipping_std',  '3 to 5 business days'),
]
keys = list(CORPUS)
print(len(keys), 'chunks,', len(QA), 'questions')

Now the core function. With probability **P** we include the golden chunk alongside distractors; with probability **1 - P** we drop it and keep only distractors. The target is a **cited chain-of-thought**, not a bare answer.

In [ ]:
rng = np.random.default_rng(0)

def make_example(question, golden_key, answer, P=0.7, n_distract=2):
    include_golden = rng.random() < P
    distractors = [k for k in keys if k != golden_key]
    chosen = list(rng.choice(distractors, size=n_distract, replace=False))
    if include_golden:
        chosen.append(golden_key)
    rng.shuffle(chosen)
    context = [CORPUS[k] for k in chosen]
    if include_golden:
        cot = 'Reason: ##begin_quote## ' + CORPUS[golden_key] + ' ##end_quote## -> ' + answer
    else:
        cot = 'Reason: context lacks the answer; from domain knowledge -> ' + answer
    return {'question': question, 'context': context, 'has_golden': include_golden, 'target': cot}

ex = make_example(*QA[0])
for k, v in ex.items():
    print(k, '::', v)

Notice the two shapes of target: when the golden chunk is present the model is taught to **quote it verbatim** (the `##begin_quote##` markers) and derive the answer; when it is absent the model must answer from **memorized** domain knowledge. Training on both is what makes RAFT robust.

## 2. Why P matters: a stylized robustness simulation

At inference the retriever returns the golden chunk only with some **recall** `r`. There are two competing skills, and `P` trades between them:

- **Use context** when the golden chunk *is* retrieved — you only get good at this by training with the golden present, so `u(P)` **increases** with `P`.
- **Fall back on memory** when it *is not* — you only get good at this from distractor-only examples, so `m(P)` **increases** with `1 - P`.

`acc = r * u(P) + (1 - r) * m(P)`. With concave `u(P) = 1 - (1-P)^2` and `m(P) = 1 - P^2`, the optimum is interior — it lands at **P* = r**. (A deliberately simple model; it isolates the tradeoff.)

In [ ]:
def accuracy(P, r):
    u = 1.0 - (1.0 - P)**2     # skill at USING retrieved context (needs golden-present training)
    m = 1.0 - P**2             # skill at MEMORY fallback (needs distractor-only training)
    return r * u + (1.0 - r) * m

for r in [1.0, 0.7, 0.4]:
    best_P = max(np.linspace(0, 1, 101), key=lambda P: accuracy(P, r))
    print('recall=' + str(r), '-> best P ~', round(best_P, 2), ' acc=', round(accuracy(best_P, r), 2))

The best `P` **tracks the retrieval recall** (`P* = r`): with perfect retrieval you want `P = 1` (always practice with the golden chunk), and as recall drops the optimum moves **below 1** so the model also learns to answer from memory. Neither extreme is right — that is RAFT's core finding.

### Visualize it

In [ ]:
Ps = np.linspace(0, 1, 101)
plt.figure()
for r in [1.0, 0.7, 0.4]:
    acc = [accuracy(P, r) for P in Ps]
    plt.plot(Ps, acc, label='retrieval recall = ' + str(r))
    pstar = Ps[int(np.argmax(acc))]
    plt.axvline(pstar, color='#334155', ls='--', lw=1)
plt.xlabel('P  (fraction of examples that include the golden document)')
plt.ylabel('simulated answer accuracy')
plt.title('RAFT: the best P tracks how often retrieval actually succeeds')
plt.legend()
plt.show()

**What to notice.** Each curve **peaks at an interior P** (the dashed line, at `P = r`), not at 0 or 1. Always-golden training (`P = 1`) only wins when retrieval is perfect; the more your retriever misses, the more distractor-only training you want. Real systems live at recall < 1, so a **moderate P** wins.

## 3. The library way

In practice you generate RAFT examples like the above (Berkeley's `gorilla/raft` provides a generator), format them as instruction/response pairs, and fine-tune with a PEFT/LoRA SFT trainer. The sketch below is the shape of that pipeline (not executed here — it needs a GPU and a base model):

In [ ]:
sketch = '''
from datasets import Dataset
from trl import SFTTrainer
from peft import LoraConfig

# 1. Build RAFT rows: {'prompt': question + context, 'completion': cited_cot}
rows = [{'prompt': fmt(ex), 'completion': ex['target']} for ex in raft_examples]
ds = Dataset.from_list(rows)

# 2. LoRA-sized SFT on the RAG task
trainer = SFTTrainer(
    model='meta-llama/Llama-3.2-3B',
    train_dataset=ds,
    peft_config=LoraConfig(r=16, lora_alpha=32, task_type='CAUSAL_LM'),
)
trainer.train()
'''
print(sketch)

## 4. Tradeoffs & when to use it

| | RAG | Fine-tuning | **RAFT** |
|---|---|---|---|
| Fresh knowledge | re-index | retrain | retrieval fresh; retrain to refresh baked facts |
| Cites sources | yes | no | yes |
| Robust to bad retrieval | no | yes | yes |
| Domain style/vocab | no | yes | yes |
| Setup cost | low | high | high |

**Use it** for a fixed, high-value domain corpus where base-model RAG is not accurate enough. **Skip it** when documents churn constantly or you need broad open-domain coverage.

**Failure modes:** `P = 1` teaches copy-paste; weak (off-topic) distractors teach nothing; bare-answer targets lose the cite-the-evidence behavior; a stale model drifts from an updated corpus.

## 5. Your turn

Add a **hard-distractor** option: draw distractors from the retriever's top-k (topically similar) rather than uniformly at random. Implement `make_example_hard` so distractors share a topic prefix with the golden key (e.g. `refund_*` when the golden is `refund_window`).

In [ ]:
def make_example_hard(question, golden_key, answer, P=0.7, n_distract=2):
    topic = golden_key.split('_')[0]
    # TODO(you): prefer distractors whose key starts with `topic`; fall back to others.
    hard = None  # <- replace
    raise NotImplementedError

# ---- checker (uncomment once implemented) ----
# ex = make_example_hard(*QA[0], P=1.0, n_distract=1)
# assert any(k.startswith('refund') for k in [] )  # see solution
print('implement make_example_hard, then check against the solution below')

<details><summary>Solution</summary>

```python
def make_example_hard(question, golden_key, answer, P=0.7, n_distract=2):
    topic = golden_key.split('_')[0]
    same = [k for k in keys if k != golden_key and k.startswith(topic)]
    other = [k for k in keys if k != golden_key and not k.startswith(topic)]
    pool = (same + other)[:max(n_distract, len(same))]
    chosen = list(rng.choice(pool, size=min(n_distract, len(pool)), replace=False))
    if rng.random() < P:
        chosen.append(golden_key)
    return {'question': question, 'context': [CORPUS[k] for k in chosen]}
```

Hard distractors force the model to *discriminate* within a topic instead of spotting an obviously off-topic chunk — the whole point of the distractor slice.
</details>

## 6. Key takeaways

- RAFT = **fine-tune on the RAG task**: golden + distractor context, cited chain-of-thought targets.
- A fraction `1 - P` of examples are **distractor-only**, forcing memorization and robustness to retrieval misses.
- The best `P` drops **below 1** as retrieval recall falls — real systems want some distractor-only training.
- Best for a **fixed domain corpus**; usually done with LoRA.

Next: [LLM Evaluation](https://ml-viz-ruby.vercel.app/courses/building-with-llms/08-llm-evaluation) to measure faithfulness before/after RAFT.